# prompt2tube — talking-head render (InfiniteTalk on RunPod)

Topic or script -> voice -> lip-synced video. Run the cells top to bottom.

**Launch:** deploy a Pod with an **80GB GPU (A100 / H100)** for full speed — the notebook
auto-switches to the fast un-quantized model on big cards. Smaller cards still work but run
the slow quantized fallback. Template: any *RunPod PyTorch 2.x*. Open JupyterLab, upload this
notebook, run it. Drop a portrait photo into the input folder when cell 5 asks.

Pod bills per second while it exists — **stop it when done**.

In [ ]:
TOPIC  = "why short-form video is taking over marketing"   # only used if SCRIPT is empty
SCRIPT = ("Most people never think about what happens in the ocean after the sun goes down. "
          "But every night, the largest migration on Earth takes place in complete darkness. "
          "Billions of tiny creatures rise from the deep water toward the surface to feed, "
          "and then sink back down before morning. The ocean at night is not empty or quiet. "
          "It is the busiest place on our entire planet.")

VOICE        = "en-US-ChristopherNeural"   # any edge-tts voice
RESOLUTION   = "infinitetalk-480"          # or infinitetalk-720
SAMPLE_STEPS = 4                           # 4 fast / 8 sharper / 40 max
USE_TEACACHE = True
FULL_BF16    = "auto"                      # auto = full model on >=46GB cards, quantized below
GPU_COST_PER_HR = 2.0                      # your pod rate, for the cost line

HF_TOKEN = ""                              # optional — only helps if HF throttles big downloads
GEMINI_API_KEY = ""                        # only needed for TOPIC mode

import os
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
BASE = "/workspace" if os.path.isdir("/workspace") else "/root"
W, INP = f"{BASE}/weights", f"{BASE}/input"
os.makedirs(INP, exist_ok=True)
print("config ok, base:", BASE)

In [ ]:
%cd {BASE}
!git clone https://github.com/MeiGen-AI/InfiniteTalk 2>/dev/null || echo "already cloned"
%cd InfiniteTalk

# repo deps minus torch/flash (keep the pod's CUDA build), plus what the image lacks
!grep -viE 'torch|flash' requirements.txt > req_min.txt || true
!pip install -q -r req_min.txt
!pip install -q librosa soundfile einops omegaconf ftfy edge-tts pyloudnorm -U huggingface_hub
# InfiniteTalk wants the older sci stack; pin it last so nothing overrides it
!pip install -q "numpy==1.26.4" "scipy==1.13.1" "soxr<0.5"
# xformers matched to this pod's torch (plain pip grabs a mismatched build)
!pip install -q xformers --index-url https://download.pytorch.org/whl/cu128
!which ffmpeg >/dev/null || (apt-get -qq update && apt-get -qq install -y ffmpeg)

import importlib
for m in ["torch","transformers","diffusers","librosa","soundfile","einops","omegaconf","xformers","pyloudnorm"]:
    assert importlib.util.find_spec(m), f"missing: {m}"
print("setup done")

In [ ]:
import torch, subprocess
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
cap  = torch.cuda.get_device_capability(0)
FULL = (vram >= 46) if FULL_BF16 == "auto" else FULL_BF16
QUANT = None if FULL else ("fp8" if cap >= (8, 9) else "int8")
print(f"{vram:.0f}GB -> " + ("full bf16 (fast)" if FULL else f"{QUANT} quantized (fits, but slow)"))

In [ ]:
# make the 2025-era repo run on current python / libraries / any GPU
import re
R = f"{BASE}/InfiniteTalk"

def sub(path, old, new, tag):
    s = open(path).read()
    if old not in s:
        print(tag, "ok"); return
    open(path, "w").write(s.replace(old, new)); print(tag, "patched")

# no int8 T5 ships with the repo -> keep T5 in bf16, only the DiT is quantized
sub(f"{R}/wan/multitalk.py",
    "quant=quant,\n            quant_dir=os.path.dirname(quant_dir) if quant_dir is not None else None,",
    "quant=None,\n            quant_dir=None,", "t5")
# ArgSpec was removed from inspect in py3.11
sub(f"{R}/wan/multitalk.py", "from inspect import ArgSpec", "# ArgSpec removed", "argspec")
# we feed our own wav, skip the bundled TTS (needs espeak)
sub(f"{R}/generate_infinitetalk.py", "from kokoro import KPipeline", "KPipeline = None", "kokoro")
# wav2vec needs eager attention on new transformers
sub(f"{R}/generate_infinitetalk.py",
    "Wav2Vec2Model.from_pretrained(wav2vec, local_files_only=True)",
    'Wav2Vec2Model.from_pretrained(wav2vec, local_files_only=True, attn_implementation="eager")', "wav2vec")

# send every attention call through the wrapper and drop the flash-only version= kwarg
for f in ["wan/modules/model.py", "wan/modules/multitalk_model.py", "wan/modules/clip.py"]:
    p = f"{R}/{f}"; s = open(p).read()
    s = s.replace("from .attention import flash_attention, attention", "from .attention import flash_attention")
    s = s.replace("from .attention import flash_attention", "from .attention import flash_attention, attention")
    s = s.replace("flash_attention(", "attention(")
    s = re.sub(r",\s*version=\d+", "", s)
    open(p, "w").write(s)
print("attention rerouted")

# use xformers (cutlass op works on every GPU arch) instead of naive sdpa
p = f"{R}/wan/modules/attention.py"; s = open(p).read()
if "cutlass" not in s:
    s = re.sub(r"out = torch\.nn\.functional\.scaled_dot_product_attention\([^)]*\)",
        "out = xformers.ops.memory_efficient_attention("
        "q.transpose(1,2).to(torch.float16), k.transpose(1,2).to(torch.float16), "
        "v.transpose(1,2).to(torch.float16), p=dropout_p, "
        "attn_bias=(xformers.ops.LowerTriangularMask() if causal else None), "
        "op=xformers.ops.MemoryEfficientAttentionCutlassOp).to(q.dtype).transpose(1,2)", s)
    open(p, "w").write(s); print("attention xformers")
# per-head chunks so the ref-attention map fits any VRAM
sub(f"{R}/wan/utils/multitalk_utils.py",
    "    _, seq_lens, heads, _ = visual_q.shape\n",
    "    _, seq_lens, heads, _ = visual_q.shape\n    split_num = heads\n", "attnmap")
print("patches done")

In [ ]:
import os
def sh(c): print("+", c); assert os.system(c) == 0, "download failed"

if QUANT is None:
    # full model: the complete Wan base incl. DiT shards
    if not os.path.exists(f"{W}/InfiniteTalk/single/infinitetalk.safetensors"):
        sh(f"hf download Wan-AI/Wan2.1-I2V-14B-480P --local-dir {W}/Wan2.1-I2V-14B-480P")
        sh(f"hf download TencentGameMate/chinese-wav2vec2-base --local-dir {W}/chinese-wav2vec2-base")
        sh(f"hf download TencentGameMate/chinese-wav2vec2-base model.safetensors --revision refs/pr/1 --local-dir {W}/chinese-wav2vec2-base")
        sh(f'hf download MeiGen-AI/InfiniteTalk --include "single/*" --local-dir {W}/InfiniteTalk')
        sh(f"hf download Kijai/WanVideo_comfy Wan21_T2V_14B_lightx2v_cfg_step_distill_lora_rank32.safetensors --local-dir {W}/lora")
    else:
        print("weights cached")
else:
    # quantized: skip the 28GB DiT shards, the quant file replaces them
    if not os.path.exists(f"{W}/InfiniteTalk/quant_models/infinitetalk_single_{QUANT}.safetensors"):
        for c in [
          f'hf download Wan-AI/Wan2.1-I2V-14B-480P --include "models_t5_umt5-xxl-enc-bf16.pth" --include "Wan2.1_VAE.pth" --include "models_clip_open-clip-xlm-roberta-large-vit-huge-14.pth" --include "config.json" --include "diffusion_pytorch_model.safetensors.index.json" --include "google/*" --include "xlm-roberta-large/*" --local-dir {W}/Wan2.1-I2V-14B-480P',
          f"hf download TencentGameMate/chinese-wav2vec2-base --local-dir {W}/chinese-wav2vec2-base",
          f"hf download TencentGameMate/chinese-wav2vec2-base model.safetensors --revision refs/pr/1 --local-dir {W}/chinese-wav2vec2-base",
          f'hf download MeiGen-AI/InfiniteTalk --include "single/*" --include "quant_models/infinitetalk_single_{QUANT}.*" --local-dir {W}/InfiniteTalk',
          f"hf download Kijai/WanVideo_comfy Wan21_T2V_14B_lightx2v_cfg_step_distill_lora_rank32.safetensors --local-dir {W}/lora"]:
            sh(c)
    else:
        print("weights cached")
os.system(f"du -sh {W}/* 2>/dev/null")
print("upload your photo to", f"{INP}/photo.jpg")

In [ ]:
if SCRIPT.strip():
    final_script = SCRIPT.strip()
else:
    import requests
    r = requests.post(
        f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?key={GEMINI_API_KEY}",
        json={"contents": [{"parts": [{"text": f"Write a 40-60 word spoken script for a short talking-head video about: {TOPIC}. Plain text only."}]}]},
        timeout=60)
    r.raise_for_status()
    final_script = r.json()["candidates"][0]["content"]["parts"][0]["text"].strip()
print(final_script)
print(f"\n~{len(final_script.split())/2.6:.0f}s of speech")

In [ ]:
# voice stage — swap this out for the cloned-voice engine later, the renderer takes any wav
with open(f"{INP}/script.txt", "w") as f: f.write(final_script)
!edge-tts --voice {VOICE} --file {INP}/script.txt --write-media {INP}/audio.mp3
!ffmpeg -y -loglevel error -i {INP}/audio.mp3 -ar 16000 -ac 1 {INP}/audio.wav
import librosa
dur = librosa.get_duration(path=f"{INP}/audio.wav")
print(f"audio: {dur:.1f}s (~{max(1, round(dur*25/81))} chunks)")

In [ ]:
import json, os, time
assert os.path.exists(f"{INP}/photo.jpg"), "upload a photo to " + f"{INP}/photo.jpg first"
json.dump({"prompt": "a person giving a friendly talk to camera, natural gestures",
           "cond_video": f"{INP}/photo.jpg",
           "cond_audio": {"person1": f"{INP}/audio.wav"}},
          open(f"{BASE}/input.json", "w"))

if QUANT is None:
    quant_flags, mem_flags = "", "--offload_model False"
else:
    quant_flags = f"--quant {QUANT} --quant_dir {W}/InfiniteTalk/quant_models/infinitetalk_single_{QUANT}.safetensors "
    mem_flags = "--offload_model False --num_persistent_param_in_dit 6000000000"

%cd {BASE}/InfiniteTalk
cmd = ("PYTORCH_ALLOC_CONF=expandable_segments:True PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True "
       "python generate_infinitetalk.py "
       f"--ckpt_dir {W}/Wan2.1-I2V-14B-480P --wav2vec_dir {W}/chinese-wav2vec2-base "
       f"--infinitetalk_dir {W}/InfiniteTalk/single/infinitetalk.safetensors {quant_flags}"
       f"--lora_dir {W}/lora/Wan21_T2V_14B_lightx2v_cfg_step_distill_lora_rank32.safetensors --lora_scale 1.0 "
       f"--input_json {BASE}/input.json --size {RESOLUTION} --sample_steps {SAMPLE_STEPS} "
       f"--sample_text_guide_scale 1 --sample_audio_guide_scale 2 --motion_frame 9 --mode streaming "
       f"{mem_flags}{' --use_teacache' if USE_TEACACHE else ''} --save_file {BASE}/result")

t0 = time.time()
!{cmd}
mins = (time.time() - t0) / 60
print(f"\n=== {mins:.1f} min -> ${mins/60*GPU_COST_PER_HR:.3f} at ${GPU_COST_PER_HR}/hr ===")

In [ ]:
from IPython.display import Video, display
import glob, os
outs = sorted(glob.glob(f"{BASE}/result*.mp4"), key=os.path.getmtime)
assert outs, "no output — check the render log above"
display(Video(outs[-1], embed=True, width=560))
print(outs[-1])

## Notes

- The **voice** cell is a drop-in slot — a cloned-voice engine replaces it without touching the render.
- Quality knobs: `SAMPLE_STEPS` (4 draft / 8 default / 40 max) and `USE_TEACACHE`.
- Weights land in `/workspace` — attach a network volume and they persist across pods.
- **Stop the pod when you're done.**